# Single Gene Deletion study of extended yeast-GEM models

- Simulate SGKO for extended genome-scale metabolic models related to yeast-GEM version 9.1.0
- Models have been adapted to YKO collection strains and generated using Jupyter notebooks (yeast9_RBA), via f2xba Python framework.
- Optimization performed using gurobipy interface, which if significantly faster than CobraPy
- Genetic background of BY4743 has been configured: MATa/MATα his3Δ1/his3Δ1 leu2Δ0/leu2Δ0 met15Δ0/MET15 lys2Δ0/LYS2 ura3Δ0/ura3Δ0
- Simulations were performed on Python 3.13, using f2xba 1.11.0, and Gurobi 13.0.3.

Peter Schubert, Heinrich-Heine University Duesseldorf, Institute for Computational Cell Biology (M. Lercher), Nov 11, 2026

## Initial setup

In [1]:
# Required imports and model names
import os
import pandas as pd

from f2xba import FbaOptimization, EcmOptimization, RbaOptimization

pd.set_option('io.excel.xlsx.reader', 'calamine')

# directories and model names (retrieve from yeast910_RBA project)
sbml_dir = 'SBML_models/yeast910'
optim_data_dir = 'data_optim/yeast910'
plots_dir = 'plots/yeast910'
results_dir = 'results/yeast910'

# models
fba_updated_model = 'yeast910_updated'
gecko_turnup_model = 'yeast910_predicted_GECKO'
gecko_model = 'yeast910_predicted_fit_GECKO'
rba_model = 'yeast910_RBA' 
tfa_model = 'yeast910_TFA'
tgecko_model = 'yeast910_TGECKO'
trba_model = 'yeast910_TRBA'

medium_fname = 'yeast9_BY4743_ypd_medium.xlsx' 
print(f'working directory: {os.getcwd()}')

working directory: /Users/peter/Documents/CCB/work/my_publications/Deya/notebooks


In [2]:
models = {fba_updated_model:  {'type': 'FBA', 'td': False, 'dir': sbml_dir},
          gecko_turnup_model: {'type': 'ECM', 'td': False, 'dir': sbml_dir},
          gecko_model:        {'type': 'ECM', 'td': False, 'dir': sbml_dir},
          rba_model:          {'type': 'RBA', 'td': False, 'dir': sbml_dir},
#          rba_fitted_model:   {'type': 'RBA', 'td': False, 'dir': sbml_dir},
          tfa_model:          {'type': 'FBA', 'td': True,  'dir': sbml_dir},
          tgecko_model:       {'type': 'ECM', 'td': True,  'dir': sbml_dir},
          trba_model:         {'type': 'RBA', 'td': True,  'dir': sbml_dir},}

remap_solution_names = {fba_updated_model: 'yeast910_FBA', gecko_turnup_model: 'yeast910_TurNuP_GECKO', 
                        gecko_model: 'yeast910_GECKO'}
analysis_names= {'moma': 'MOMA', 'linear moma': 'lMOMA', 'room': 'ROOM', 'linear room': 'lROOM'}

In [3]:
# load ypd medium used of SGKO analysis
fname = os.path.join(optim_data_dir, medium_fname)
with pd.ExcelFile(fname) as xlsx:
    ypd_medium = pd.read_excel(xlsx, sheet_name='ypd_medium', index_col=0)['max_uptake'].to_dict()
print(f'YPD medium ({len(ypd_medium)} nutrients) loaded from {fname}')

# Metabolite concentrations to be fixed with a margin (Buckstein, 2007)
nt_concs_µmol_per_l = {
    'atp_c': 3560, 'adp_c': 116, 'datp_c': 181, 'ctp_c': 325, 'dctp_c': 184, 'gtp_c': 1660, 
    'gdp_c': 203, 'dgtp_c': 92, 'utp_c': 667, 'udp_c': 54, 'dttp_c': 256, 'accoa_c': 1390,
    'atp_m': 3560, 'adp_m': 116, 'datp_m': 181, 'ctp_m': 325, 'dctp_m': 184, 'gtp_m': 1660, 
    'gdp_m': 203, 'dgtp_m': 92, 'utp_m': 667, 'dttp_m': 256, 'accoa_m': 1390}
factor = 2.0
metabolite_molar_concs = {sid: (µmol_per_l * 1e-6/factor, µmol_per_l*1e-6*factor) 
                          for sid, µmol_per_l in  nt_concs_µmol_per_l.items()}

YPD medium (42 nutrients) loaded from ../../yeast9_RBA/notebooks_yeast910/data_optim/yeast9_BY4743_ypd_medium.xlsx


## SGKO for extended models
- SGKO optimization is performed for selected genome scale metabolic models using f2xba modeling framework
    - optimization for genes not used in the wild type solutions are considered to have not growth impact
- SGKO results are exported. 
- MOMA, linear MOMA, ROOM and linear ROOM methods can be applied to FBA and GECKO models (Note: simulating ROOM takes a long time)

In [4]:
# FBA and GECKO type SGKO simulations 
# genetic background of BW25311 wrt MG1655 ∆araBAD ∆rhaBAD ∆lacZ
# b0063\araB, b0062\araA, b0061\araD, b3904\rhaB, b3903\rhaA, b3902\rhaD, b0344\lacZ

# [fba_updated_model, gecko_turnup_model, gecko_model, tfa_model, tgecko_model, rba_model, trba_model]

model_names =  [fba_updated_model, gecko_turnup_model, gecko_model, tfa_model, tgecko_model, rba_model, trba_model]
special_methods =  ['linear moma', 'moma', 'linear room', 'room']# 'linear moma', 'moma', 'linear room', 'room'] 

strain_background = {'YOR202W', 'YCL018W', 'YEL021W' }

for model_name in model_names:
    meta_data = models[model_name]
    if meta_data['type'] == 'FBA':
        xo = FbaOptimization(os.path.join(meta_data['dir'], f'{model_name}.xml'))
        xo.medium = ypd_medium
    elif meta_data['type'] == 'ECM':
        xo = EcmOptimization(os.path.join(meta_data['dir'], f'{model_name}.xml'))
        xo.medium = {rid: 1000.0 for rid in ypd_medium}
    elif meta_data['type'] == 'RBA':
        xo = RbaOptimization(os.path.join(meta_data['dir'], f'{model_name}.xml'))
        sigma = xo.avg_enz_saturation
        rel_mmol_per_l =  (sigma / (1.0-min(sigma, .99))) * xo.importer_km
        xo.medium = {re.sub('EX_', '', ex_ridx): rel_mmol_per_l for ex_ridx in ypd_medium}
    
    # fix metabolite concentrations for TD models
    if meta_data['td'] is True:
        orig_concs = xo.set_tfa_metab_concentrations(metabolite_molar_concs)
        print(f'{len(orig_concs)} metabolite concentrations constrained')
    
    # set strain background
    orig_bounds = {}
    if strain_background is not None:
        orig_bounds.update(xo.gene_knock_outs(strain_background))
        print(f'Genetic background, genes deleted: {strain_background}')
    
    if meta_data['type'] == 'RBA':
        df_sgko = xo.single_gene_deletion(gr_min=0.01, gr_min_alt=0.02, gr_max=0.7, bisection_tol=.5e-3)

    else:
        # baseline model sgko analysis
        df_sgko = xo.single_gene_deletion()
        
        # special methods MOMA and ROOM for FBA and GECKO
        if model_name in {fba_updated_model, gecko_model}:
            for special_method in special_methods:
                print(f'  {special_method} on {model_name}')
                if 'moma' in special_method:
                    df_sp_sgko = xo.single_gene_deletion(method=special_method)
                elif 'room' in special_method:
                    df_sp_sgko = xo.single_gene_deletion(method=special_method, delta=0.03, epsilon=1e-3, time_limit=30.0)
                remapped_name = remap_solution_names.get(model_name, model_name)
                sp_fname = os.path.join(results_dir, f'sgko_{remapped_name}_{analysis_names[special_method]}.xlsx')        
                with pd.ExcelWriter(sp_fname) as writer:
                    df_sp_sgko.to_excel(writer, sheet_name='sgko')
                    
    # export SGKO results (excluding special methods)
    remapped_name = remap_solution_names.get(model_name, model_name)
    fname = os.path.join(results_dir, f'sgko_{remapped_name}.xlsx')
    with pd.ExcelWriter(fname) as writer:
        df_sgko.to_excel(writer, sheet_name='sgko')

SBML model loaded by sbmlxdf: ../../yeast9_RBA/notebooks_yeast910/SBML_models/yeast910_updated.xml (Wed Nov  4 08:23:28 2026)
Set parameter Username
Set parameter LicenseID to value 2835868
Academic license - for non-commercial use only - expires 2027-06-16
LP Model of yeastGEM_v9_1_0
4182 variables, 2787 constraints, 15732 non-zero matrix coefficients
Genetic background, genes deleted: {'YEL021W', 'YOR202W', 'YCL018W'}
614 genes potentially active given flux distribution.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1156/1156 [01:03<00:00, 18.23it/s]


  linear moma on yeast910_updated
616 genes potentially active given flux distribution.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1156/1156 [03:02<00:00,  6.33it/s]


  moma on yeast910_updated
616 genes potentially active given flux distribution.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1156/1156 [03:16<00:00,  5.87it/s]


... (20 outputs left out)

SBML model loaded by sbmlxdf: ../../yeast9_RBA/notebooks_yeast910/SBML_models/yeast910_RBA.xml (Wed Nov  4 08:52:57 2026)
LP Model of yeastGEM_v9_1_0_RBA
10352 variables, 8926 constraints, 113453 non-zero matrix coefficients
3697 enzymes, 13 process machines
RBA enzyme efficiency constraints configured (C_EF_xxx, C_ER_xxx) ≤ 0
Genetic background, genes deleted: {'YEL021W', 'YOR202W', 'YCL018W'}
680 genes potentially active given flux distribution.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1384/1384 [32:35<00:00,  1.41s/it]


SBML model loaded by sbmlxdf: ../../yeast9_RBA/notebooks_yeast910/SBML_models/yeast910_TRBA.xml (Wed Nov  4 09:06:38 2026)
MILP Model of yeastGEM_v9_1_0_TFA_RBA
16475 variables, 14278 constraints, 132820 non-zero matrix coefficients
3697 enzymes, 13 process machines, 1237 TD reaction constraints
RBA enzyme efficiency constraints configured (C_EF_xxx, C_ER_xxx) ≤ 0
23 metabolite concentrations constrained
Genetic background, genes deleted: {'YEL021W', 'YOR202W', 'YCL018W'}
684 genes potentially active given flux distribution.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1384/1384 [6:13:58<00:00, 16.21s/it]
